# 02-2. 조건 분기와 Multi-Route Workflow

- 예상 시간: 150분
- 선수 실습: 02-1_langgraph_state_node_edge.ipynb
- 실습 난이도: 중급
- 핵심 기술: Conditional Edge, Router, 규칙 기반 Multi-Route 구조
- 최종 산출물: 입력 유형별로 처리 Node를 선택하는 LangGraph Workflow
- 버전: 학생용 실습파일 (`TODO`가 표시된 코드 셀을 완성해야 이후 셀이 정상 동작합니다)

## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. Router와 Conditional Edge의 역할을 설명할 수 있다.
2. Routing과 Tool Selection의 차이를 구분할 수 있다.
3. 규칙 기반으로 입력 유형별 처리 Node를 연결하는 Multi-Route Graph를 구성할 수 있다.
4. 지원하지 않는 입력과 오류 상황을 안전하게 처리할 수 있다.

## 2. 문제 상황

Notebook 02-1은 어떤 입력이 들어와도 항상 `analyze_input → generate_answer` 순서로
실행되는 고정 Workflow였다. 그러나 실제 서비스에서는 "오늘 날씨 어때?"와 "문서에서
온보딩 관련 내용 찾아줘"는 전혀 다른 처리와 Tool이 필요하다. 이 모든 경우를 하나의
Node 안에 조건문으로 욱여넣으면 Node의 책임이 뒤섞이고 유지보수가 어려워진다.
LangGraph의 Conditional Edge와 Multi-Route 구조로 이 문제를 해결한다. 02-1번의 고정 Edge에 조건 분기 하나를 추가하는 것이 이번 실습의 핵심이며, 시각화와 호출 제한 코드는 선택 학습 요소다. 이 Notebook에서는 개발자가 작성한 규칙이 처리 Node를 선택한다. LLM이 Tool을 선택하는 Agent 구조는 02-3에서 다룬다.

## 3. 핵심 개념

### 3.1 정의

Router는 현재 State를 기준으로 다음에 실행할 Node를 결정하는 함수이다. Conditional
Edge는 그 Router 함수의 반환값을 실제 Node 이름과 연결하는 LangGraph의 실행 경로이며,
고정 Edge와 달리 실행할 때마다 다른 경로로 이어질 수 있다.

### 3.2 개념이 필요한 이유

모든 입력이 동일한 처리 과정을 거치지 않는 경우, 고정 Edge만으로는 Workflow를
표현하기 어렵다. 질문에는 바로 답하고, 문서 검색 요청에는 문서를 뒤지고, 계산
요청에는 계산기를 돌려야 한다면, 입력을 먼저 분류하고 그 결과에 따라 다른 Node로
보내는 구조가 필요하다.

### 3.3 주요 구성요소

| 구성요소 | 역할 |
|---|---|
| `input_classifier` (Node) | 입력을 분석해 `route` 값을 State에 기록 |
| `route_input` (Router 함수) | State의 `route` 값을 읽어 경로 키를 반환 |
| `add_conditional_edges` | Router 반환값과 실제 Node 이름을 매핑 |
| `path_map` | 경로 키 → Node 이름 매핑 딕셔너리 |
| 유형별 Node | 개발자가 연결한 실제 작업 함수를 수행 |
| `result_generator` | 어떤 경로를 거쳤든 최종 응답 형식을 통일 |

### 3.4 동작 과정

```text
START
  ↓
input_classifier
  ├─ question    → question_node
  ├─ document    → document_node
  ├─ calculation → calculation_node
  └─ unsupported → unsupported_node
           ↓
      result_generator
           ↓
          END
```

### 3.5 코드와 개념의 대응 관계

| 코드 요소 | 구현 개념 |
|---|---|
| `classify_input()` | 입력 분류 (Router가 사용할 판단 값 계산) |
| `route_input()` | Router 함수 |
| `add_conditional_edges(start, router_fn, path_map)` | Conditional Edge 연결 |
| `path_map` 딕셔너리 | 라우팅 키 ↔ 실행 Node 이름 매핑 |
| `result_generator` | 여러 경로의 결과를 하나의 형식으로 합류 |

### 3.6 유사 개념과의 차이

**Routing vs Tool Selection**

```text
Routing
→ 다음에 실행할 Node를 선택

Tool Selection
→ 특정 Node 안에서 실행할 기능을 선택
```

| 구분 | Routing | Tool Selection |
|---|---|---|
| 결정 시점 | Node 실행 전, Edge 단계 | Node 실행 중 |
| 결정 대상 | 다음에 실행할 Node | 호출할 함수(Tool) |
| 구현 위치 | Conditional Edge, Router 함수 | Node 내부 코드 |
| 이 Notebook의 예 | `route_input()`이 네 가지 경로 중 선택 | 선택된 Node가 정해진 함수를 직접 호출 |

**규칙 기반 Router vs LLM 기반 Router**

| 구분 | 규칙 기반 Router | LLM 기반 Router |
|---|---|---|
| 판단 방식 | 조건문(`if`/`elif`)으로 직접 분류 | LLM에게 분류를 요청 |
| 실행 속도·비용 | 빠르고 API 비용 없음 | API 호출 비용과 지연 발생 |
| 예측 가능성 | 동일한 입력 → 항상 동일한 결과 | 동일한 입력이라도 응답이 달라질 수 있음 |
| 이 Notebook 사용 방식 | `classify_input()`에서 사용 | 분류 규칙으로 처리하기 어려운 유형이 계속 늘어날 때 고려 |

### 3.7 사용 시점과 적용 조건

입력 유형에 따라 처리 로직이나 Tool이 달라질 때 Conditional Edge를 사용한다. 모든
입력이 항상 같은 순서로 처리된다면 Notebook 02-1처럼 고정 Edge로 충분하다.

### 3.8 한계와 주의사항

- Router가 반환하는 값이 `path_map`에 없는 키라면 실행 중 오류가 발생한다.
- 분기가 많아질수록 모든 경로가 실제로 `END`까지 도달하는지 하나하나 확인해야 한다.

### 3.9 자주 발생하는 오해

"Conditional Edge가 Tool을 실행한다"는 오해가 있다. 실제로는 Conditional Edge는
다음 Node를 선택할 뿐이며, 실제 Tool 실행은 선택된 Node 내부 코드가 담당한다.

### 3.10 핵심 정리

- Router는 State를 보고 다음 Node를 결정하고, Conditional Edge는 그 결정을 실제
  실행 경로로 연결한다.
- Routing은 개발자가 정의한 판단값으로 "어떤 Node로 갈지" 결정한다. 이 Notebook의
  함수 호출은 미리 연결되어 있으며, LLM 기반 Tool Selection과는 다르다.
- Router가 반환하는 값은 반드시 `path_map`에 정의된 키와 일치해야 하며, 지원하지
  않는 입력을 위한 기본 경로가 필요하다.

## 4. 실행 구조

```text
build_multitool_graph()
   │
   ├─ StateGraph(RouterState) 생성
   ├─ add_node("input_classifier", classify_input)
   ├─ add_node("question_node" / "document_node" / "calculation_node" / "unsupported_node")
   ├─ add_node("result_generator", result_generator)
   ├─ add_edge(START, "input_classifier")
   ├─ add_conditional_edges("input_classifier", route_input, path_map)
   ├─ add_edge(각 유형별 Node, "result_generator")
   ├─ add_edge("result_generator", END)
   └─ compile() → 실행 가능한 Graph

graph.invoke(initial_state) → route에 따라 다른 경로를 거친 최종 State
```

## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

from agentic_ai.config import get_settings
from agentic_ai.logging_utils import save_log
from agentic_ai.models import get_chat_model
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.tools import calculate, search_keyword

settings = get_settings()
print_environment_summary(settings, needs_chat_model=True)


## 6. 최소 실행 예제

`RouterState`를 정의하기 전에, Conditional Edge의 동작만 가장 단순한 형태로 먼저
확인한다. 숫자 하나를 받아 짝수/홀수에 따라 다른 Node로 분기한다.

In [ ]:
class ParityState(TypedDict):
    user_input: str
    route: str
    final_answer: str


def classify_parity(state: ParityState) -> dict:
    """숫자의 짝수/홀수 여부로 route를 정한다."""
    return {"route": "even" if int(state["user_input"]) % 2 == 0 else "odd"}


def route_parity(state: ParityState) -> str:
    return state["route"]


def even_node(state: ParityState) -> dict:
    return {"final_answer": f"{state['user_input']}은(는) 짝수입니다."}


def odd_node(state: ParityState) -> dict:
    return {"final_answer": f"{state['user_input']}은(는) 홀수입니다."}


mini_builder = StateGraph(ParityState)
mini_builder.add_node("classify_parity", classify_parity)
mini_builder.add_node("even_node", even_node)
mini_builder.add_node("odd_node", odd_node)
mini_builder.add_edge(START, "classify_parity")
mini_builder.add_conditional_edges(
    "classify_parity", route_parity, {"even": "even_node", "odd": "odd_node"}
)
mini_builder.add_edge("even_node", END)
mini_builder.add_edge("odd_node", END)
mini_graph = mini_builder.compile()



In [ ]:
# 시각화는 네트워크 환경에 따라 실패할 수 있으므로 선택적으로 실행한다.
from IPython.display import Image, display

try:
    display(Image(mini_graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("그래프 시각화를 건너뜁니다:", exc)

In [ ]:
print(mini_graph.invoke({"user_input": "4", "route": "", "final_answer": ""}))
print(mini_graph.invoke({"user_input": "7", "route": "", "final_answer": ""}))

## 7. 단계별 구현

### 7.1 State 정의

여러 Node가 공유할 `RouterState`를 정의한다. `route`는 어떤 경로로 분기했는지,
`tool_log`는 어떤 Tool을 호출했는지 기록한다. `MAX_TOOL_CALLS`는 한 번 실행에서
허용하는 최대 Tool 호출 횟수이다.

In [ ]:
class RouterState(TypedDict):
    user_input: str
    route: str
    tool_log: list[str]
    result: str
    final_answer: str
    error: str | None


MAX_TOOL_CALLS = 3

### 7.2 입력 분류 Node

**TODO**: `classify_input()`을 작성한다.

- `user_input`에 `"문서"`와 (`"찾아줘"` 또는 `"검색"`)이 모두 포함되어 있으면
  `"document"`를 반환한다.
- 숫자가 포함되어 있고, 사칙연산 기호(`+ - * /`) 또는 `"더하기"`, `"빼기"`,
  `"곱하기"`, `"나누기"` 중 하나가 포함되어 있으면 `"calculation"`을 반환한다.
- 위 조건에 해당하지 않고 `"?"`, `"까요"`, `"나요"`로 끝나면 `"question"`을
  반환한다.
- 그 외에는 `"unsupported"`를 반환한다.
- 반환값은 `{"route": ...}` 형태의 딕셔너리여야 한다.

예상 출력: `classify_input({"user_input": "128 나누기 4는 얼마야?", ...})` →
`{"route": "calculation"}`

In [ ]:
def classify_input(state: RouterState) -> dict:
    """규칙 기반으로 입력 유형을 분류한다."""
    text = state["user_input"]
    # TODO: 1) text에 "문서"와 ("찾아줘" 또는 "검색")이 모두 포함되어 있으면
    #       route = "document"로 정한다.
    # TODO: 2) 그렇지 않고 숫자가 포함되어 있으면서 사칙연산 기호(+ - * /) 또는
    #       "더하기"/"빼기"/"곱하기"/"나누기" 중 하나가 포함되어 있으면
    #       route = "calculation"으로 정한다.
    # TODO: 3) 그렇지 않고 "?", "까요", "나요"로 끝나면 route = "question"으로
    #       정한다.
    # TODO: 4) 그 외에는 route = "unsupported"로 정한다.
    # TODO: 5) {"route": route} 형태의 딕셔너리를 반환한다.
    raise NotImplementedError("TODO: classify_input을 완성하세요.")


print(classify_input({"user_input": "128 나누기 4는 얼마야?", "route": "", "tool_log": [], "result": "", "final_answer": "", "error": None}))


### 7.3 Router 함수

Router 함수는 `input_classifier` Node가 이미 계산해 둔 `route` 값을 그대로
읽어서 반환한다. 분류 로직과 라우팅 결정을 분리해 두면, 나중에 분류 방식만
(예: 규칙 기반 → LLM 기반) 바꾸더라도 Router 함수와 `path_map`은 그대로 재사용할
수 있다.

In [ ]:
def route_input(state: RouterState) -> str:
    """Conditional Edge가 사용할 Router 함수. 이미 계산된 route 값을 그대로 반환한다."""
    return state["route"]

### 7.4 유형별 처리 Node와 함수 호출

각 Node는 자신이 담당하는 유형에 맞는 함수를 개발자가 미리 연결해 직접 호출한다.
`tool_log`에 어떤 함수를 호출했는지 남기고, `MAX_TOOL_CALLS`를 넘기면 더 이상
호출하지 않는다. 여기서는 LLM이 여러 Tool 중 하나를 선택하지 않는다는 점에 주의한다.

In [ ]:
_OP_WORDS = {"더하기": "+", "빼기": "-", "곱하기": "*", "나누기": "/"}


def extract_expression(text: str) -> str:
    """자연어 계산 요청에서 사칙연산 수식만 추출한다."""
    cleaned = text
    for word, symbol in _OP_WORDS.items():
        cleaned = cleaned.replace(word, f" {symbol} ")
    allowed = set("0123456789+-*/. ")
    cleaned = "".join(ch for ch in cleaned if ch in allowed)
    return cleaned.strip()


def extract_keyword_query(text: str) -> str:
    """"문서에서 '키워드' 찾아줘" 형식에서 키워드만 추출한다."""
    parts = text.split("'")
    if len(parts) >= 2:
        return parts[1]
    return text


SAMPLE_DOCUMENT = (
    "리모트워크 만족도는 74%로 높았다. "
    "다만 신입 온보딩과 화상회의 운영에 대한 보완 요청이 있었다."
)


def calculation_node(state: RouterState) -> dict:
    tool_log = list(state.get("tool_log", []))
    if len(tool_log) >= MAX_TOOL_CALLS:
        return {"error": "최대 Tool 호출 횟수를 초과했습니다.", "result": ""}
    expression = extract_expression(state["user_input"])
    try:
        value = calculate(expression)
        tool_log.append(f"calculate({expression!r}) -> {value}")
        return {"result": f"계산 결과: {value}", "tool_log": tool_log, "error": None}
    except ValueError as exc:
        tool_log.append(f"calculate({expression!r}) -> 오류")
        return {"result": "", "tool_log": tool_log, "error": str(exc)}


def document_node(state: RouterState) -> dict:
    tool_log = list(state.get("tool_log", []))
    if len(tool_log) >= MAX_TOOL_CALLS:
        return {"error": "최대 Tool 호출 횟수를 초과했습니다.", "result": ""}
    keyword = extract_keyword_query(state["user_input"])
    found = search_keyword(SAMPLE_DOCUMENT, keyword)
    tool_log.append(f"search_keyword({keyword!r}) -> {found['found']}")
    if found["found"]:
        start = max(0, found["position"] - 5)
        snippet = SAMPLE_DOCUMENT[start:found["position"] + 15]
        result = f"'{keyword}' 관련 내용을 문서에서 찾았습니다: ...{snippet}..."
    else:
        result = f"'{keyword}'와 관련된 내용을 문서에서 찾지 못했습니다."
    return {"result": result, "tool_log": tool_log, "error": None}


def question_node(state: RouterState) -> dict:
    tool_log = list(state.get("tool_log", []))
    model = get_chat_model()
    response = model.invoke(f"다음 질문에 한 문장으로 답하라: {state['user_input']}")
    tool_log.append("llm_direct_answer")
    return {"result": response.content, "tool_log": tool_log, "error": None}


def unsupported_node(state: RouterState) -> dict:
    tool_log = list(state.get("tool_log", []))
    tool_log.append("none")
    return {
        "result": "지원하지 않는 요청입니다. 질문, 문서 검색, 계산 요청만 처리할 수 있습니다.",
        "tool_log": tool_log,
        "error": None,
    }


# MAX_TOOL_CALLS 동작 확인: tool_log가 이미 가득 찬 상태로 호출하면 Tool을 실행하지 않는다.
full_state = {
    "user_input": "5 더하기 5는?",
    "route": "calculation",
    "tool_log": ["calculate('1+1') -> 2.0"] * MAX_TOOL_CALLS,
    "result": "",
    "final_answer": "",
    "error": None,
}
print(calculation_node(full_state))

### 7.5 결과 통합 Node

경로가 무엇이었든, `result_generator`가 최종 응답 형식을 하나로 통일한다.

In [ ]:
def result_generator(state: RouterState) -> dict:
    """어떤 경로를 거쳤는지와 상관없이 최종 답변 형식을 통일한다."""
    if state.get("error"):
        final = f"[오류] {state['error']}"
    else:
        final = f"[{state['route']}] {state['result']}"
    return {"final_answer": final}

### 7.6 Graph 조립

**TODO**: `build_multitool_graph()`를 작성한다.

- `StateGraph(RouterState)`로 builder를 만든다.
- `"input_classifier"`, `"question_node"`, `"document_node"`, `"calculation_node"`,
  `"unsupported_node"`, `"result_generator"`를 `add_node()`로 등록한다.
- `add_edge(START, "input_classifier")`로 시작 경로를 연결한다.
- `add_conditional_edges("input_classifier", route_input, path_map)`으로 4가지
  경로를 연결한다. `path_map`은 `{"question": "question_node", "document":
  "document_node", "calculation": "calculation_node", "unsupported":
  "unsupported_node"}`이다.
- 4개 유형별 Node에서 각각 `"result_generator"`로 향하는 Edge를 연결한다.
- `add_edge("result_generator", END)`로 마무리한다.
- `compile()` 결과를 반환한다.

In [ ]:
def build_multitool_graph():
    """입력 유형에 따라 다른 처리 Node를 선택하는 Graph를 만든다."""
    # TODO: 1) StateGraph(RouterState)로 builder를 만든다.
    # TODO: 2) "input_classifier"(classify_input), "question_node", "document_node",
    #       "calculation_node", "unsupported_node", "result_generator"를
    #       add_node()로 등록한다.
    # TODO: 3) add_edge(START, "input_classifier")로 시작 경로를 연결한다.
    # TODO: 4) add_conditional_edges("input_classifier", route_input, path_map)으로
    #       4가지 경로를 연결한다. path_map은 {"question": "question_node",
    #       "document": "document_node", "calculation": "calculation_node",
    #       "unsupported": "unsupported_node"}이다.
    # TODO: 5) 4개 유형별 Node에서 각각 "result_generator"로 향하는 Edge를 연결한다.
    # TODO: 6) add_edge("result_generator", END)로 마무리하고 builder.compile()
    #       결과를 반환한다.
    raise NotImplementedError("TODO: build_multitool_graph를 완성하세요.")


multitool_graph = build_multitool_graph()
print(multitool_graph)


In [ ]:
# 시각화는 네트워크 환경에 따라 실패할 수 있으므로 선택적으로 실행한다.
from IPython.display import Image, display

try:
    display(Image(multitool_graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("그래프 시각화를 건너뜁니다:", exc)

### 7.7 Invoke

In [ ]:
sample_state = {
    "user_input": "128 나누기 4는 얼마야?",
    "route": "",
    "tool_log": [],
    "result": "",
    "final_answer": "",
    "error": None,
}
sample_result = multitool_graph.invoke(sample_state)
print(sample_result)

## 8. 실행 결과 관찰

일반 질문 2개, 문서 분석 2개, 계산 요청 2개, 지원하지 않는 입력 2개로 전체 흐름을
확인한다.

In [ ]:
TEST_INPUTS = [
    "오늘 날씨 어때?",
    "리모트워크의 장점이 뭐야?",
    "문서에서 '온보딩' 찾아줘",
    "문서에서 '화상회의' 검색해줘",
    "128 나누기 4는 얼마야?",
    "25 더하기 17 계산해줘",
    "노래 한 곡 불러줘",
    "",
]

test_results = []
for text in TEST_INPUTS:
    state = {
        "user_input": text,
        "route": "",
        "tool_log": [],
        "result": "",
        "final_answer": "",
        "error": None,
    }
    outcome = multitool_graph.invoke(state)
    test_results.append(outcome)
    print(f"입력: {text!r:35} | route: {outcome['route']:12} | 답변: {outcome['final_answer']}")

**결과 해석**: `route` 값이 입력마다 다르게 정해지고, 선택된 Node가 미리 연결된
함수를 직접 호출한다. `tool_log`를 보면 어떤 함수가 몇 번 호출됐는지 실행마다
독립적으로 기록된다는 것을 확인할 수 있다. LLM 기반 Tool Selection은 다음 02-3에서 비교한다.

## 9. 실패 실험: path_map에 없는 경로

Router 함수가 `path_map`에 정의되지 않은 값을 반환하면 어떤 일이 벌어지는지
확인한다.

In [ ]:
def broken_route_input(state: RouterState) -> str:
    """route 값에 오타가 있는 잘못된 Router 예시 ("document" -> "documents")."""
    route = state["route"]
    return "documents" if route == "document" else route


broken_builder = StateGraph(RouterState)
broken_builder.add_node("input_classifier", classify_input)
broken_builder.add_node("question_node", question_node)
broken_builder.add_node("document_node", document_node)
broken_builder.add_node("calculation_node", calculation_node)
broken_builder.add_node("unsupported_node", unsupported_node)
broken_builder.add_node("result_generator", result_generator)
broken_builder.add_edge(START, "input_classifier")
broken_builder.add_conditional_edges(
    "input_classifier",
    broken_route_input,
    {
        "question": "question_node",
        "document": "document_node",
        "calculation": "calculation_node",
        "unsupported": "unsupported_node",
    },
)
broken_builder.add_edge("question_node", "result_generator")
broken_builder.add_edge("document_node", "result_generator")
broken_builder.add_edge("calculation_node", "result_generator")
broken_builder.add_edge("unsupported_node", "result_generator")
broken_builder.add_edge("result_generator", END)
broken_graph = broken_builder.compile()



In [ ]:
try:
    broken_graph.invoke(
        {
            "user_input": "문서에서 '온보딩' 찾아줘",
            "route": "",
            "tool_log": [],
            "result": "",
            "final_answer": "",
            "error": None,
        }
    )
    print("오류 없이 실행됨 (예상과 다름)")
except Exception as exc:
    print("실패:", type(exc).__name__, exc)

**원인 분석 질문**

- `classify_input()`은 정확히 `"document"`를 반환했는가?
- 오류는 분류 단계에서 발생했는가, 아니면 Router가 반환한 값을 `path_map`에서
  찾는 단계에서 발생했는가?
- `path_map`에 있는 키를 하나라도 잘못 적으면 어떤 입력이 영향을 받는가?

## 10. 오류 수정 실습

`broken_route_input` 대신 원래의 `route_input`을 사용하면 문제가 해결된다는 것을
재확인한다. Router 함수는 `path_map`에 정의된 키와 정확히 같은 값만 반환해야
한다.

In [ ]:
fixed_builder = StateGraph(RouterState)
fixed_builder.add_node("input_classifier", classify_input)
fixed_builder.add_node("question_node", question_node)
fixed_builder.add_node("document_node", document_node)
fixed_builder.add_node("calculation_node", calculation_node)
fixed_builder.add_node("unsupported_node", unsupported_node)
fixed_builder.add_node("result_generator", result_generator)
fixed_builder.add_edge(START, "input_classifier")
fixed_builder.add_conditional_edges(
    "input_classifier",
    route_input,
    {
        "question": "question_node",
        "document": "document_node",
        "calculation": "calculation_node",
        "unsupported": "unsupported_node",
    },
)
fixed_builder.add_edge("question_node", "result_generator")
fixed_builder.add_edge("document_node", "result_generator")
fixed_builder.add_edge("calculation_node", "result_generator")
fixed_builder.add_edge("unsupported_node", "result_generator")
fixed_builder.add_edge("result_generator", END)
fixed_graph = fixed_builder.compile()

fixed_outcome = fixed_graph.invoke(
    {
        "user_input": "문서에서 '온보딩' 찾아줘",
        "route": "",
        "tool_log": [],
        "result": "",
        "final_answer": "",
        "error": None,
    }
)
print(fixed_outcome)
assert fixed_outcome["route"] == "document"
print("재검증 통과")

## 11. 도전 과제

1. `"summary"` 유형을 추가해서, "요약해줘"가 포함된 입력을 별도 Node로 분기해본다.
2. `classify_input()`을 규칙 기반 대신 LLM 기반으로 바꿔보고, 같은 입력에도
   응답이 달라질 수 있는지 관찰한다.
3. `calculation_node`에서 `calculate()`가 `ValueError`를 던지는 입력(예: "10 나누기
   0은?")을 넣어 오류 경로가 `result_generator`까지 안전하게 전달되는지 확인한다.

## 12. 테스트

**테스트 유형: 로컬 Graph 통합 테스트 — 결정적, 외부 API 호출 없음**

규칙 기반 Router, Tool 함수와 전체 Graph 경로를 함께 검증한다.

In [ ]:
question_state = {"user_input": "오늘 날씨 어때?", "route": "", "tool_log": [], "result": "", "final_answer": "", "error": None}
assert classify_input(question_state)["route"] == "question"

document_state = {"user_input": "문서에서 '온보딩' 찾아줘", "route": "", "tool_log": [], "result": "", "final_answer": "", "error": None}
assert classify_input(document_state)["route"] == "document"

calculation_state = {"user_input": "128 나누기 4는 얼마야?", "route": "", "tool_log": [], "result": "", "final_answer": "", "error": None}
assert classify_input(calculation_state)["route"] == "calculation"

unsupported_state = {"user_input": "노래 한 곡 불러줘", "route": "", "tool_log": [], "result": "", "final_answer": "", "error": None}
assert classify_input(unsupported_state)["route"] == "unsupported"

assert extract_expression("128 나누기 4는 얼마야?").replace(" ", "") == "128/4"
assert extract_keyword_query("문서에서 '온보딩' 찾아줘") == "온보딩"

full_state_check = {"user_input": "1+1", "route": "calculation", "tool_log": ["x"] * MAX_TOOL_CALLS, "result": "", "final_answer": "", "error": None}
assert calculation_node(full_state_check)["error"] is not None

graph_instance = build_multitool_graph()
assert graph_instance is not None

calc_result = graph_instance.invoke(calculation_state)
assert calc_result["route"] == "calculation"
assert "32" in calc_result["final_answer"]

doc_result = graph_instance.invoke(document_state)
assert doc_result["route"] == "document"
assert doc_result["error"] is None

unsupported_result = graph_instance.invoke(unsupported_state)
assert unsupported_result["route"] == "unsupported"

print("테스트 통과")

## 13. 결과 저장

In [ ]:
routing_log = {
    "test_inputs": TEST_INPUTS,
    "test_results": [
        {"user_input": r["user_input"], "route": r["route"], "tool_log": r["tool_log"], "final_answer": r["final_answer"]}
        for r in test_results
    ],
}
saved_path = save_log(routing_log, OUTPUT_DIR / "logs" / "06_routing_multitool_log.json")
print("저장 위치:", saved_path)

## 14. 핵심 정리

- Router는 State를 보고 다음 Node를 결정하고, Conditional Edge는 그 결정을 실제
  실행 경로로 연결한다.
- Routing은 "어떤 Node로 갈지"를 결정한다. 이 Notebook의 함수 호출은 개발자가
  미리 연결하며, LLM 기반 Tool Selection과는 다르다.
- 규칙 기반 Router는 빠르고 예측 가능하지만, 분류 규칙으로 다루기 어려운 유형이
  많아지면 LLM 기반 Router를 고려할 수 있다.
- 모든 분기는 결국 `END`에 도달해야 하며, 지원하지 않는 입력을 위한 기본 경로가
  반드시 필요하다.
- Router가 반환하는 값이 `path_map`에 없으면 오류 없이 넘어가지 않고 실행 중
  오류로 즉시 드러난다.

## 15. 확인 문제

1. Router와 Conditional Edge는 각각 어떤 역할을 하는가?
2. 이 Notebook의 규칙 기반 Routing과 LLM 기반 Tool Selection은 무엇이 다른가?
3. 규칙 기반 Router와 LLM 기반 Router는 각각 어떤 상황에서 유리한가?
4. Router가 `path_map`에 없는 값을 반환하면 어떤 일이 벌어지는가? 이를 예방하려면
   어떻게 작성해야 하는가?